# SageMaker Daily Prediction

Fetches the latest market data for Crude Oil (CL=F) and Gold (GC=F) futures via `yfinance`,
calls 6 SageMaker endpoints (3 models x 2 tickers), and generates BUY/SELL/HOLD trading signals.

**Signal composition:** 70% model majority vote + 15% EMA + 15% McClellan oscillator

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import talib
import yfinance as yf
import boto3

REGION = "us-east-1"
BUCKET = "labs-usd-01"
PREFIX = "AAI_540_group_8"
ROLE_ARN = "arn:aws:iam::806081623304:role/LabRole"
INFERENCE_IMAGE = (
    "763104351884.dkr.ecr.us-east-1.amazonaws.com/"
    "pytorch-inference:2.1.0-cpu-py310-ubuntu20.04-sagemaker"
)

TICKERS = ["CL=F", "GC=F"]
MODEL_NAMES = ["lstm", "transformer", "bilstm_attention"]
MODEL_LABELS = {"lstm": "LSTM", "transformer": "Transformer", "bilstm_attention": "BiLSTM-Attention"}

FEATURE_COLUMNS = [
    "high", "low", "open", "volume",
    "MA", "EMA", "KAMA", "WMA", "MidPrice",
    "BOP", "CMO", "MFI", "ROC", "WILLR",
    "AD", "OBV", "NATR", "ATR", "TRANGE", "TSF",
]

LOOKBACK = 20
EMA_SPAN = 10

sm_client = boto3.client("sagemaker", region_name=REGION)
runtime_client = boto3.client("sagemaker-runtime", region_name=REGION)

## Step 1: Deploy or Reuse Endpoints

In [ ]:
def get_model_package_arn(group_name, model_name):
    """Get the latest approved model package ARN matching model_name."""
    resp = sm_client.list_model_packages(
        ModelPackageGroupName=group_name,
        ModelApprovalStatus="Approved",
        SortBy="CreationTime",
        SortOrder="Descending",
    )
    for pkg in resp["ModelPackageSummaryList"]:
        pkg_detail = sm_client.describe_model_package(ModelPackageName=pkg["ModelPackageArn"])
        meta = pkg_detail.get("CustomerMetadataProperties", {})
        if meta.get("model_architecture") == model_name:
            return pkg["ModelPackageArn"]
    raise ValueError(f"No package found for {model_name} in {group_name}")


def endpoint_exists(endpoint_name):
    """Check if an endpoint exists and is InService."""
    try:
        resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
        return resp["EndpointStatus"] == "InService"
    except sm_client.exceptions.ClientError:
        return False


def create_endpoint_from_package(endpoint_name, model_package_arn):
    """Create model + config + endpoint from a model package."""
    model_name = f"{endpoint_name}-model"
    config_name = f"{endpoint_name}-config"

    try:
        sm_client.create_model(
            ModelName=model_name,
            ExecutionRoleArn=ROLE_ARN,
            Containers=[{"ModelPackageName": model_package_arn}],
        )
    except sm_client.exceptions.ClientError as e:
        if "Cannot create already existing" not in str(e):
            raise

    try:
        sm_client.create_endpoint_config(
            EndpointConfigName=config_name,
            ProductionVariants=[{
                "VariantName": "AllTraffic",
                "ModelName": model_name,
                "InstanceType": "ml.m5.large",
                "InitialInstanceCount": 1,
            }],
        )
    except sm_client.exceptions.ClientError as e:
        if "Cannot create already existing" not in str(e):
            raise

    try:
        sm_client.create_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=config_name,
        )
    except sm_client.exceptions.ClientError as e:
        if "Cannot create already existing" not in str(e):
            raise

In [ ]:
groups = {"CL=F": "futures-cl-models", "GC=F": "futures-gc-models"}
endpoint_names = []

for ticker in TICKERS:
    ticker_safe = ticker.replace("=", "").replace("F", "").lower()
    for model_name in MODEL_NAMES:
        ep_name = f"futures-{ticker_safe}-{model_name.replace('_', '-')}"
        endpoint_names.append(ep_name)

        if endpoint_exists(ep_name):
            print(f"  {ep_name}: Already InService")
        else:
            print(f"  {ep_name}: Deploying...")
            pkg_arn = get_model_package_arn(groups[ticker], model_name)
            create_endpoint_from_package(ep_name, pkg_arn)

# Wait for all to be InService
print("\nWaiting for endpoints...")
pending = set(endpoint_names)
while pending:
    for name in list(pending):
        try:
            resp = sm_client.describe_endpoint(EndpointName=name)
            status = resp["EndpointStatus"]
            if status == "InService":
                pending.remove(name)
            elif status == "Failed":
                raise RuntimeError(f"{name} failed: {resp.get('FailureReason')}")
        except sm_client.exceptions.ClientError:
            pass
    if pending:
        print(f"  Still waiting on {len(pending)} endpoints...")
        time.sleep(30)

print("All endpoints are InService!")

## Step 2: Fetch Latest Market Data

In [ ]:
market_data = {}
for ticker in TICKERS:
    df = yf.download(ticker, period="120d", auto_adjust=True)

    # Flatten MultiIndex columns if present
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [c.lower() for c in df.columns]
    market_data[ticker] = df
    print(f"{ticker}: {len(df)} rows, {df.index[0].date()} to {df.index[-1].date()}")
    print(f"  Latest close: ${df['close'].iloc[-1]:.2f}")

In [ ]:
def compute_talib_features(ohlcv_df):
    """Compute TA-Lib features from OHLCV data."""
    df = ohlcv_df.copy().sort_index()

    df["MA"] = talib.MA(df["close"], timeperiod=10)
    df["EMA"] = talib.EMA(df["close"], timeperiod=10)
    df["KAMA"] = talib.KAMA(df["close"], timeperiod=10)
    df["WMA"] = talib.WMA(df["close"], timeperiod=10)
    df["MidPrice"] = talib.MIDPRICE(df["high"], df["low"], timeperiod=10)

    df["BOP"] = talib.BOP(df["open"], df["high"], df["low"], df["close"])
    df["CMO"] = talib.CMO(df["close"], timeperiod=10)
    df["MFI"] = talib.MFI(df["high"], df["low"], df["close"], df["volume"])
    df["ROC"] = talib.ROC(df["close"], timeperiod=10)
    df["WILLR"] = talib.WILLR(df["high"], df["low"], df["close"], timeperiod=14)

    df["AD"] = talib.AD(df["high"], df["low"], df["close"], df["volume"])
    df["OBV"] = talib.OBV(df["close"], df["volume"])

    df["NATR"] = talib.NATR(df["high"], df["low"], df["close"], timeperiod=14)
    df["ATR"] = talib.ATR(df["high"], df["low"], df["close"], timeperiod=14)
    df["TRANGE"] = talib.TRANGE(df["high"], df["low"], df["close"])

    df["TSF"] = talib.TSF(df["close"], timeperiod=14)

    df = df.iloc[15:]
    return df


featured_data = {}
for ticker in TICKERS:
    featured_data[ticker] = compute_talib_features(market_data[ticker])
    print(f"{ticker}: {len(featured_data[ticker])} rows after feature engineering")

## Step 3: Generate Predictions & Signals

In [ ]:
def ema_update(prev, value, span):
    """Incremental EMA update."""
    alpha = 2.0 / (span + 1)
    return value if prev is None else alpha * value + (1 - alpha) * prev


signals = {}

for ticker in TICKERS:
    ticker_safe = ticker.replace("=", "").replace("F", "").lower()
    df = featured_data[ticker]

    # Extract last LOOKBACK rows of features
    features = df[FEATURE_COLUMNS].values[-LOOKBACK:]
    current_close = float(df["close"].iloc[-1])
    yesterday_close = float(df["close"].iloc[-2])

    # --- Model predictions ---
    predictions = {}
    votes = 0
    for model_name in MODEL_NAMES:
        ep_name = f"futures-{ticker_safe}-{model_name.replace('_', '-')}"
        payload = json.dumps({"features": features.tolist()})
        resp = runtime_client.invoke_endpoint(
            EndpointName=ep_name,
            ContentType="application/json",
            Body=payload,
        )
        result = json.loads(resp["Body"].read().decode())
        pred_price = result["predicted_price"]
        predictions[model_name] = pred_price
        if pred_price > yesterday_close:
            votes += 1

    model_signal = 1.0 if votes >= 2 else -1.0

    # --- EMA signal ---
    closes = df["close"].values[-EMA_SPAN * 2:]
    ema_val = None
    for c in closes:
        ema_val = ema_update(ema_val, float(c), EMA_SPAN)
    ema_signal = 1.0 if current_close > ema_val else -1.0

    # --- McClellan oscillator ---
    recent_closes = df["close"].values[-40:]
    mcclel_fast_val = None
    mcclel_slow_val = None
    prev = None
    for c in recent_closes:
        c = float(c)
        change = 0.0 if prev is None else c - prev
        prev = c
        mcclel_fast_val = ema_update(mcclel_fast_val, change, 19)
        mcclel_slow_val = ema_update(mcclel_slow_val, change, 39)
    mcclellan = mcclel_fast_val - mcclel_slow_val
    mcclellan_signal = 1.0 if mcclellan > 0 else -1.0

    # --- Combined signal ---
    combined = 0.70 * model_signal + 0.15 * ema_signal + 0.15 * mcclellan_signal

    if combined > 0:
        recommendation = "BUY"
    elif combined < 0:
        recommendation = "SELL"
    else:
        recommendation = "HOLD"

    signals[ticker] = {
        "current_price": current_close,
        "yesterday_close": yesterday_close,
        "predictions": predictions,
        "model_signal": model_signal,
        "ema_value": ema_val,
        "ema_signal": ema_signal,
        "mcclellan_value": mcclellan,
        "mcclellan_signal": mcclellan_signal,
        "combined_signal": combined,
        "recommendation": recommendation,
    }

## Step 4: Trading Signal Summary

In [ ]:
for ticker in TICKERS:
    s = signals[ticker]
    print("=" * 60)
    print(f"  {ticker}")
    print("=" * 60)
    print(f"  Current Price:    ${s['current_price']:.2f}")
    print(f"  Yesterday Close:  ${s['yesterday_close']:.2f}")
    print()

    # Model predictions
    print("  Model Predictions:")
    for mname in MODEL_NAMES:
        pred = s["predictions"][mname]
        direction = "UP" if pred > s["yesterday_close"] else "DOWN"
        arrow = "^" if direction == "UP" else "v"
        print(f"    {MODEL_LABELS[mname]:20s}  ${pred:>10.2f}  {arrow} {direction}")

    print()
    print("  Signal Components:")
    print(f"    Model Vote (70%):     {'+1 BUY' if s['model_signal'] > 0 else '-1 SELL'}")
    print(f"    EMA-{EMA_SPAN} (15%):       {'+1 BUY' if s['ema_signal'] > 0 else '-1 SELL'}  (EMA={s['ema_value']:.2f})")
    print(f"    McClellan (15%):      {'+1 BUY' if s['mcclellan_signal'] > 0 else '-1 SELL'}  (osc={s['mcclellan_value']:.4f})")
    print()
    print(f"  Combined Score:   {s['combined_signal']:+.2f}")
    print(f"  >>> RECOMMENDATION: {s['recommendation']} <<<")
    print()

## Step 5: CLEANUP — Delete Endpoints

**IMPORTANT:** Run this cell when you are done to avoid ongoing charges.
Each `ml.m5.large` endpoint costs ~$0.115/hour. Six endpoints = ~$0.69/hour.

In [ ]:
for ticker in TICKERS:
    ticker_safe = ticker.replace("=", "").replace("F", "").lower()
    for model_name in MODEL_NAMES:
        ep_name = f"futures-{ticker_safe}-{model_name.replace('_', '-')}"

        try:
            sm_client.delete_endpoint(EndpointName=ep_name)
            print(f"Deleted endpoint: {ep_name}")
        except Exception as e:
            print(f"  Skip endpoint {ep_name}: {e}")

        try:
            sm_client.delete_endpoint_config(EndpointConfigName=f"{ep_name}-config")
            print(f"Deleted config: {ep_name}-config")
        except Exception as e:
            print(f"  Skip config: {e}")

        try:
            sm_client.delete_model(ModelName=f"{ep_name}-model")
            print(f"Deleted model: {ep_name}-model")
        except Exception as e:
            print(f"  Skip model: {e}")

print("\nAll endpoints cleaned up!")